# 1. Overview

This notebook runs the `samples` workflow as a thin operator console. It restores the required runtime inputs, runs the public `workflows.samples` workflow, and publishes the workflow outputs to Drive.

Workflow logic lives in `text_to_sign_production.workflows.samples`; this notebook only configures, reviews, executes, and summarizes the public workflow contract.


# 2. Operator Configuration

Set the repository revision, runtime roots, requested splits, gates config path, and person-selection policy for this run. These values are reviewed before runtime setup or workflow execution.


## 2.1 Repository and roots

Configure the repository checkout and runtime/Drive roots used by setup, restore, and publish operations.


In [ ]:
from pathlib import Path


REPO_URL = "https://github.com/0xmillennium/text-to-sign-production.git"
REPO_REF = "chore/core-layout-notebook-workflows"
PROJECT_ROOT = Path("/content/text-to-sign-production")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/text-to-sign-production")


## 2.2 Samples workflow inputs

Configure the samples workflow inputs that are passed into the public workflow config.


In [ ]:
SAMPLES_SPLITS = ("train", "val", "test")
GATES_CONFIG_RELATIVE_PATH = Path("configs/data/gates.yaml")
PERSON_SELECTION_POLICY = "highest_canonical_signal"


## 2.3 Configuration review

Review the selected values before preparing the runtime.


In [ ]:
print(f"Repository URL: {REPO_URL}")
print(f"Repository ref: {REPO_REF}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Drive project root: {DRIVE_PROJECT_ROOT}")
print(f"Samples splits: {SAMPLES_SPLITS}")
print(f"Gates config: {GATES_CONFIG_RELATIVE_PATH}")
print(f"Person selection policy: {PERSON_SELECTION_POLICY}")


# 3. Runtime Setup

Prepare the Colab runtime only: mount Drive, ensure system tooling, check out the repository, install dependencies, and make `src` importable. Workflow restore, execution, validation, and publish steps start later.


## 3.1 Mount Drive

Mount Google Drive and confirm the configured Drive parent exists.


In [ ]:
from google.colab import drive


drive.mount("/content/drive", force_remount=False)
if not DRIVE_PROJECT_ROOT.parent.is_dir():
    raise FileNotFoundError(f"Drive MyDrive root is missing: {DRIVE_PROJECT_ROOT.parent}")
print(f"Drive mounted: {DRIVE_PROJECT_ROOT.parent}")


## 3.2 System packages

Ensure `zstd` is available for tar.zst restore and publish archive operations.


In [ ]:
import shutil


if shutil.which("zstd") is None:
    !sudo apt-get update
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to update apt package index.")

    !sudo apt-get install -y zstd
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to install zstd.")
    print("Installed zstd.")
else:
    print("zstd is already available.")

!zstd --version
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("zstd command is not available after preflight.")


## 3.3 Repository checkout

Clone the requested repository revision into the configured runtime project root.


In [ ]:
%cd /content

if PROJECT_ROOT.exists():
    print(f"Removing stale repository checkout: {PROJECT_ROOT}")
    !rm -rf {PROJECT_ROOT}
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(f"Failed to remove existing directory: {PROJECT_ROOT}")

!git clone {REPO_URL} {PROJECT_ROOT}
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("Failed to clone repository.")

!git -C {PROJECT_ROOT} checkout {REPO_REF}
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError(f"Failed to checkout revision {REPO_REF}.")

print(f"Repository ready: {PROJECT_ROOT}")
!git -C {PROJECT_ROOT} rev-parse HEAD
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("Failed to determine checked out revision.")


## 3.4 Install dependencies

Install the Colab dependency set from the checkout. Colab may restart the runtime after package installation; if that happens, global variables are cleared, so rerun Operator Configuration and Runtime Setup cells through this point before continuing.


In [ ]:
%cd {PROJECT_ROOT}
%pip install --upgrade pip
%pip install -r "requirements-colab.txt"
print("Repository dependencies installed from requirements-colab.txt.")


## 3.5 Add source tree to import path

Run this after dependency installation and after any runtime restart. It depends on restored globals such as `PROJECT_ROOT`, so rerun the configuration cells first if Colab restarted.


In [ ]:
import sys


%cd {PROJECT_ROOT}
source_path = PROJECT_ROOT / "src"
if str(source_path) not in sys.path:
    sys.path.append(str(source_path))
print(f"Repository src directory available on sys.path: {source_path}")


# 4. Notebook Helpers

Define small notebook-local helpers for visible execution and readable summaries. These helpers do not build workflow plans, infer path ownership, validate business rules, or reach into non-public workflow internals.


## 4.1 Execution helpers

Run workflow-provided shell commands. Publish metadata materialization is handled by the public samples workflow API.


In [ ]:
def run_command_spec(spec):
    command = spec.command
    print(f"[{command.label}]")
    print(command.bash)
    !bash -o pipefail -c {command.arg}
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(command.failure)


def run_command_specs(specs):
    for spec in specs:
        run_command_spec(spec)



## 4.2 Display helpers

Print compact generic summaries without changing workflow state.


In [ ]:
def _render_text(value) -> str:
    if value is None:
        return "none"
    return str(value)


def display_block_title(title, *, leading_blank: bool = False) -> None:
    rendered_title = str(title).strip()
    if not rendered_title:
        rendered_title = "Untitled"

    if leading_blank:
        print()

    print(rendered_title)
    print("-" * len(rendered_title))


def display_lines(title, lines, *, leading_blank: bool = False) -> None:
    rendered_lines = tuple(lines)

    display_block_title(title, leading_blank=leading_blank)

    if not rendered_lines:
        print("- none")
        return

    for line in rendered_lines:
        print(f"- {_render_text(line)}")


def display_review_sections(sections, *, leading_blank: bool = False) -> None:
    rendered_sections = tuple(sections)

    if not rendered_sections:
        display_block_title("Review", leading_blank=leading_blank)
        print("- none")
        return

    for section_index, section in enumerate(rendered_sections):
        display_block_title(
            section.title,
            leading_blank=leading_blank or section_index > 0,
        )

        items = tuple(section.items)
        if not items:
            print("- none")
            continue

        for item_index, item in enumerate(items, start=1):
            if item_index > 1:
                print()

            print(f"{item_index}. {_render_text(item.label)}")

            fields = tuple(item.fields)
            if not fields:
                print("   - none")
                continue

            for field in fields:
                print(f"   {field.label}: {_render_text(field.value)}")

# 5. Workflow Setup

Import the public `workflows.samples` API, build the samples workflow config, and ask the workflow to build the restore runtime plan. The outputs of this section are `samples_config` and `samples_runtime_plan`.


## 5.1 Import public samples API

Use only the notebook-facing public workflow surface.


In [ ]:
from text_to_sign_production.workflows.samples import (
    SamplesWorkflowConfig,
    build_samples_publish_plan,
    build_samples_runtime_plan,
    materialize_samples_publish_metadata,
    run_samples_workflow,
    summarize_samples_publish_plan,
    summarize_samples_publish_plan_review,
    summarize_samples_runtime_plan_review,
    summarize_samples_workflow_outputs,
    validate_samples_inputs,
    verify_samples_runtime_inputs,
)


## 5.2 Build config and runtime plan

Construct the workflow config from operator settings and review the restore plan counts.


In [ ]:
samples_config = SamplesWorkflowConfig(
    project_root=PROJECT_ROOT,
    drive_project_root=DRIVE_PROJECT_ROOT,
    splits=SAMPLES_SPLITS,
    gates_config_relative_path=GATES_CONFIG_RELATIVE_PATH,
    person_selection_policy=PERSON_SELECTION_POLICY,
)
samples_runtime_plan = build_samples_runtime_plan(samples_config)

print(f"Project root: {samples_runtime_plan.project_root}")
print(f"Drive project root: {samples_runtime_plan.drive_project_root}")
print(f"Splits: {tuple(split.value for split in samples_config.splits)}")
print(f"Restore file transfers: {len(samples_runtime_plan.restore_file_transfers)}")
print(f"Restore archive extracts: {len(samples_runtime_plan.restore_archive_extracts)}")


# 6. Restore Inputs

Review the workflow-owned restore specs, then run one operator action that restores translations, extracts keypoint archives, verifies runtime inputs, and validates workflow inputs. This section produces a runtime ready for the samples workflow.


## 6.1 Review restore plan

Review translation restore specs and keypoint archive extract specs. This is a review-only checkpoint; it does not execute restore work.


In [ ]:
translation_restore_specs = samples_runtime_plan.restore_file_transfers
keypoint_archive_extract_specs = samples_runtime_plan.restore_archive_extracts

samples_restore_review_sections = summarize_samples_runtime_plan_review(samples_runtime_plan)
display_review_sections(samples_restore_review_sections)


## 6.2 Restore inputs and verify readiness

Operator action: execute the reviewed restore specs, then verify runtime inputs and validate workflow inputs in sequence.


In [ ]:
print("Restore plan reviewed.")
print("Running translation restores...")
run_command_specs(translation_restore_specs)
print("Translation restores complete.")

print("Extracting keypoint archives...")
run_command_specs(keypoint_archive_extract_specs)
print("Archive extraction complete.")

print("Verifying runtime inputs...")
verify_samples_runtime_inputs(samples_config)
print("Runtime inputs verified.")

print("Validating workflow inputs...")
validate_samples_inputs(samples_config)
print("Workflow inputs validated.")

print("Restore complete.")


# 7. Run Samples Workflow

Run the public samples workflow once the runtime inputs are restored and validated. This section produces `samples_result` and a review of workflow-owned output summaries.


## 7.1 Run workflow

Execute the workflow through the public API and store its result.


In [ ]:
print("Running samples workflow...")
samples_result = run_samples_workflow(samples_config)
print("Samples workflow complete.")


## 7.2 Review workflow outputs

Review the workflow output summary before building any publish plan. This subsection is review-only.


In [ ]:
samples_output_summary = summarize_samples_workflow_outputs(samples_result)

display_lines(
    "Passed counts by split",
    (
        f"{split.value}: {count}"
        for split, count in samples_output_summary.passed_counts_by_split.items()
    ),
)
display_lines(
    "Dropped counts by split",
    (
        f"{split.value}: {count}"
        for split, count in samples_output_summary.dropped_counts_by_split.items()
    ),
    leading_blank=True,
)
display_lines(
    "Untiered manifest paths",
    samples_output_summary.untiered_manifest_paths,
    leading_blank=True,
)
display_lines("Report paths", samples_output_summary.report_paths, leading_blank=True)


# 8. Publish Outputs

Build and review the public publish plan, then execute it as one operator publish action. The publish review is plan-summary data only unless the public workflow API later provides a true publish execution-result DTO.


## 8.1 Build and review publish plan

Review the planned publish targets and specs before execution. This is plan-summary data, not an execution-result summary unless the public API truly provides one.


In [ ]:
print("Building publish plan...")
samples_publish_plan = build_samples_publish_plan(samples_config, samples_result)
samples_publish_plan_summary = summarize_samples_publish_plan(samples_publish_plan)

publish_file_transfer_specs = samples_publish_plan.file_transfers
archive_create_specs = samples_publish_plan.archive_creates
archive_verify_specs = samples_publish_plan.archive_verifies
samples_publish_review_sections = summarize_samples_publish_plan_review(samples_publish_plan)

print("Publish plan ready.")
display_block_title("Publish plan summary", leading_blank=True)
print(f"- planned files: {samples_publish_plan_summary.planned_file_count}")
print(f"- planned archives: {samples_publish_plan_summary.planned_archive_count}")
display_lines(
    "Publish target paths",
    samples_publish_plan_summary.target_paths,
    leading_blank=True,
)
display_review_sections(samples_publish_review_sections, leading_blank=True)


## 8.2 Execute publish plan

Operator action: materialize archive member lists if the plan exposes them, then publish files, create archives, and verify archives in sequence.


In [ ]:
print("Preparing archive metadata...")
materialize_samples_publish_metadata(samples_publish_plan)
print("Archive metadata prepared.")

print("Publishing manifests and reports...")
run_command_specs(publish_file_transfer_specs)
print("File publish complete.")

print("Creating archives...")
run_command_specs(archive_create_specs)
print("Archive creation complete.")

print("Verifying archives...")
run_command_specs(archive_verify_specs)
print("Archive verification complete.")

print("Publish action complete.")


# 9. Final Summary

Print one compact operator summary for the requested splits, workflow output counts, output paths, and planned publish target paths. Publish summary data reflects planned outputs only.


In [ ]:
display_block_title("Final operator summary")
print(f"Requested splits: {tuple(split.value for split in samples_config.splits)}")
display_lines(
    "Passed counts by split",
    (
        f"{split.value}: {count}"
        for split, count in samples_output_summary.passed_counts_by_split.items()
    ),
    leading_blank=True,
)
display_lines(
    "Dropped counts by split",
    (
        f"{split.value}: {count}"
        for split, count in samples_output_summary.dropped_counts_by_split.items()
    ),
    leading_blank=True,
)
display_lines(
    "Untiered manifest output paths",
    samples_output_summary.untiered_manifest_paths,
    leading_blank=True,
)
display_lines("Report paths", samples_output_summary.report_paths, leading_blank=True)
display_lines(
    "Publish target paths",
    samples_publish_plan_summary.target_paths,
    leading_blank=True,
)
print("Publish summary reflects planned outputs only.")
print("The current public API does not expose a publish execution-result DTO.")
